# S3MambaDA V2 — Ready-to-Run Notebook

Corrected EEGMMIDB four-class pipeline with stable Sinc filtering, DGNN, 2-layer BiGRU, warm-up domain adaptation, SupCon, training augmentation, validation-based best epoch selection, AdaBN, and publication figures.

**Important:** the temporal module is a bidirectional GRU, not a true Mamba SSM. No diffusion augmentation is implemented.

In [ ]:
# CELL 1 - CONFIG + IMPORTS
import os, math, random, json, time, copy, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, classification_report, roc_curve, auc
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

SEED=42
DATA_DIR="./eegmmidb"
TOTAL_SUBJECTS=109
RUNS=[4,6,8,10,12,14]

TMIN=0.5
TMAX=3.5
FS=250
N_SAMPLES=int((TMAX-TMIN)*FS)

CHANNEL_MODE="FIRST22"   # FIRST22 / ALL64 / CUSTOM
CUSTOM_CHANNELS=[]

if CHANNEL_MODE=="FIRST22":
    N_CHANNELS=22
elif CHANNEL_MODE=="ALL64":
    N_CHANNELS=64
elif CHANNEL_MODE=="CUSTOM":
    if not CUSTOM_CHANNELS:
        raise ValueError("CUSTOM_CHANNELS is empty.")
    N_CHANNELS=len(CUSTOM_CHANNELS)
else:
    raise ValueError("Invalid CHANNEL_MODE.")

NUM_CLASSES=4
CLASS_NAMES=["Left Hand","Right Hand","Both Fists","Both Feet"]

NUM_FILTERS=10
SINC_KERNEL=81
SINC_MIN_FREQ=1.0
SINC_MAX_FREQ=40.0
SPATIAL_DIM=64

GRU_HIDDEN=64
GRU_LAYERS=2
GRU_DROPOUT=0.20

BATCH_SIZE=64
EPOCHS=200
WARMUP_EPOCHS=20
LEARNING_RATE=3e-4
WEIGHT_DECAY=1e-4
LABEL_SMOOTHING=0.05
DOMAIN_WEIGHT_MAX=0.15
SUPCON_WEIGHT_MAX=0.10
SUPCON_TEMP=0.07
GRAD_CLIP=1.0

VAL_FRACTION=0.10

USE_AUGMENTATION=True
AUG_NOISE_STD=0.01
AMPLITUDE_MIN=0.90
AMPLITUDE_MAX=1.10
TEMPORAL_MASK_PROB=0.30
CHANNEL_DROPOUT_PROB=0.20

EVAL_MODE="PROJECT"   # PROJECT / RANDOM / EXHAUSTIVE
CURATED_TEST_POOL=[4,15,23,29,31,42,55,71,82,95]
NUM_TEST_FOLDS=10
NUM_TRAIN_SUBJECTS=99

RESULTS_DIR=Path("./S3MambaDA_V2_RESULTS")
FIGURES_DIR=RESULTS_DIR/"FIGURES"
CHECKPOINT_DIR=RESULTS_DIR/"CHECKPOINTS"
TABLES_DIR=RESULTS_DIR/"TABLES"
for d in [RESULTS_DIR,FIGURES_DIR,CHECKPOINT_DIR,TABLES_DIR]:
    d.mkdir(parents=True,exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE=torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE=torch.device("mps")
else:
    DEVICE=torch.device("cpu")

print("Device:",DEVICE)
print("Input:",(N_CHANNELS,N_SAMPLES))
print("Evaluation:",EVAL_MODE)

In [ ]:
# CELL 2 - DATASET CHECK + CHANNEL DIAGNOSTIC
if not Path(DATA_DIR).exists():
    raise FileNotFoundError(f"Dataset not found: {Path(DATA_DIR).resolve()}")

AVAILABLE_SUBJECTS=[
    s for s in range(1,TOTAL_SUBJECTS+1)
    if (Path(DATA_DIR)/f"S{s:03d}").exists()
]
print("Available subjects:",len(AVAILABLE_SUBJECTS))

diagnostic=Path(DATA_DIR)/"S004"/"S004R04.edf"
if diagnostic.exists():
    raw=mne.io.read_raw_edf(str(diagnostic),preload=False,verbose=False)
    print("S004R04 channels:",len(raw.ch_names))
    print(raw.ch_names[:64])

In [ ]:
# CELL 3 - CORRECTED DATASET
class EEGMMIDB_Dataset(Dataset):
    def __init__(self,data_dir,subjects,runs=RUNS,tmin=TMIN,tmax=TMAX,training=False):
        self.data_dir=str(data_dir); self.subjects=list(subjects); self.runs=list(runs)
        self.tmin=tmin; self.tmax=tmax; self.training=training
        self.epochs=[]; self.labels=[]; self.subject_ids=[]; self.run_ids=[]
        self.load_data()

    def _select_channels(self,raw):
        if CHANNEL_MODE=="FIRST22": raw.pick(raw.ch_names[:22])
        elif CHANNEL_MODE=="ALL64": raw.pick(raw.ch_names[:64])
        else:
            missing=[c for c in CUSTOM_CHANNELS if c not in raw.ch_names]
            if missing: raise ValueError(f"Missing channels: {missing}")
            raw.pick(CUSTOM_CHANNELS)
        return raw

    def load_data(self):
        for s in self.subjects:
            folder=Path(self.data_dir)/f"S{s:03d}"
            if not folder.exists(): continue
            for run in self.runs:
                edf=folder/f"S{s:03d}R{run:02d}.edf"
                if not edf.exists(): continue
                try:
                    raw=mne.io.read_raw_edf(str(edf),preload=True,verbose=False)
                    raw=self._select_channels(raw)
                    raw.resample(FS,verbose=False)
                    events,event_id=mne.events_from_annotations(raw,verbose=False)
                    if "T1" not in event_id or "T2" not in event_id: continue

                    extraction={"T1":event_id["T1"],"T2":event_id["T2"]}

                    if run in [4,8,12]:
                        mapping={"T1":0,"T2":1}
                    elif run in [6,10,14]:
                        mapping={"T1":2,"T2":3}
                    else: continue

                    ep=mne.Epochs(raw,events,event_id=extraction,
                                  tmin=self.tmin,tmax=self.tmax-1/FS,
                                  baseline=None,preload=True,verbose=False)
                    data=ep.get_data()
                    codes=ep.events[:,-1]

                    for i,x in enumerate(data):
                        if int(codes[i])==event_id["T1"]: label=mapping["T1"]
                        elif int(codes[i])==event_id["T2"]: label=mapping["T2"]
                        else: continue
                        self.epochs.append(x.astype(np.float32))
                        self.labels.append(label)
                        self.subject_ids.append(s-1)
                        self.run_ids.append(run)
                except Exception as e:
                    print(f"[ERROR] S{s:03d} R{run:02d}: {type(e).__name__}: {e}")

    def _augment(self,x):
        if not self.training or not USE_AUGMENTATION: return x
        x=x*torch.empty(1).uniform_(AMPLITUDE_MIN,AMPLITUDE_MAX)
        x=x+torch.randn_like(x)*AUG_NOISE_STD
        if random.random()<TEMPORAL_MASK_PROB:
            ml=random.randint(10,min(40,max(10,x.shape[1]//5)))
            st=random.randint(0,max(0,x.shape[1]-ml))
            x[:,st:st+ml]=0
        if random.random()<CHANNEL_DROPOUT_PROB:
            x[random.randrange(x.shape[0])]=0
        return x

    def __len__(self): return len(self.epochs)

    def __getitem__(self,i):
        x=torch.tensor(self.epochs[i],dtype=torch.float32)
        y=torch.tensor(self.labels[i],dtype=torch.long)
        s=torch.tensor(self.subject_ids[i],dtype=torch.long)
        x=(x-x.mean(dim=1,keepdim=True))/(x.std(dim=1,keepdim=True)+1e-6)
        return self._augment(x),y,s

# critical validation
s004=EEGMMIDB_Dataset(DATA_DIR,[4],training=False)
print("S004 trials:",len(s004))
print("Labels:",Counter(s004.labels))
print("Runs:",Counter(s004.run_ids))
print("Shape:",s004[0][0].shape)
assert len(s004)>0 and set(s004.labels)=={0,1,2,3}
assert tuple(s004[0][0].shape)==(N_CHANNELS,N_SAMPLES)
print("✓ Dataset validation passed.")

In [ ]:
# CELL 4 - MODEL MODULES
class GradientReversalLayer(torch.autograd.Function):
    @staticmethod
    def forward(ctx,x,lamb):
        ctx.lamb=lamb; return x.view_as(x)
    @staticmethod
    def backward(ctx,g):
        return -ctx.lamb*g,None
def grl(x,lamb): return GradientReversalLayer.apply(x,lamb)

class SincFilterBank(nn.Module):
    def __init__(self):
        super().__init__()
        self.low_parameter=nn.Parameter(torch.linspace(2.0,30.0,NUM_FILTERS))
        self.band_parameter=nn.Parameter(torch.ones(NUM_FILTERS)*5.0)

    def forward(self,x):
        B,C,T=x.shape
        n=torch.arange(-(SINC_KERNEL//2),SINC_KERNEL//2+1,device=x.device,dtype=x.dtype)
        filt=[]
        for i in range(NUM_FILTERS):
            low=SINC_MIN_FREQ+torch.sigmoid(self.low_parameter[i]/10.0)*(SINC_MAX_FREQ-SINC_MIN_FREQ-2.0)
            bw=F.softplus(self.band_parameter[i])
            min_high=low+1.0
            max_high=torch.tensor(SINC_MAX_FREQ,device=x.device,dtype=x.dtype)
            high=torch.minimum(min_high+bw,max_high)
            high=torch.maximum(high,min_high)
            f1=low/FS; f2=high/FS
            k=2*f2*torch.sinc(2*f2*n)-2*f1*torch.sinc(2*f1*n)
            k=k*torch.hamming_window(SINC_KERNEL,device=x.device,dtype=x.dtype)
            k=k/(torch.sqrt(torch.sum(k*k))+1e-8)
            filt.append(k.view(1,1,-1))
        filt=torch.cat(filt,0)
        y=F.conv1d(x.reshape(B*C,1,T),filt,padding="same")
        return y.reshape(B,C,NUM_FILTERS,T).permute(0,2,1,3)

class DGNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.W_Q=nn.Linear(NUM_FILTERS,NUM_FILTERS)
        self.W_K=nn.Linear(NUM_FILTERS,NUM_FILTERS)
        self.W_V=nn.Linear(N_CHANNELS,SPATIAL_DIM)

    def forward(self,x):
        B,Fb,C,T=x.shape
        d=x.mean(dim=-1).transpose(1,2)
        Q=self.W_Q(d); K=self.W_K(d)
        A=F.softmax(torch.matmul(Q,K.transpose(-2,-1))/math.sqrt(NUM_FILTERS),dim=-1)
        I=torch.eye(C,device=x.device,dtype=x.dtype).unsqueeze(0)
        A=A+I
        deg=A.sum(-1).clamp_min(1e-6)
        D=torch.diag_embed(1/torch.sqrt(deg))
        A=D@A@D
        xt=x.permute(0,1,3,2)
        y=torch.einsum("bij,bntj->bnti",A,xt)
        return F.elu(self.W_V(y)).permute(0,1,3,2)

class TemporalEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru=nn.GRU(SPATIAL_DIM,GRU_HIDDEN,GRU_LAYERS,
                        dropout=GRU_DROPOUT,batch_first=True,bidirectional=True)
        self.proj=nn.Linear(GRU_HIDDEN*2,SPATIAL_DIM)
    def forward(self,x):
        y,_=self.gru(x.transpose(1,2))
        return self.proj(y).transpose(1,2)

class SEAttention(nn.Module):
    def __init__(self):
        super().__init__()
        r=16
        self.fc=nn.Sequential(
            nn.Linear(SPATIAL_DIM,SPATIAL_DIM//r,bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(SPATIAL_DIM//r,SPATIAL_DIM,bias=False),
            nn.Sigmoid()
        )
    def forward(self,x):
        a=self.fc(x.mean(dim=2)).unsqueeze(-1)
        return (x*a).mean(dim=2)

class S3MambaDA(nn.Module):
    def __init__(self):
        super().__init__()
        self.sinc=SincFilterBank()
        self.dgnn=DGNN()
        self.temporal=TemporalEncoder()
        self.se=SEAttention()
        self.cls=nn.Sequential(
            nn.BatchNorm1d(SPATIAL_DIM),
            nn.Dropout(0.20),
            nn.Linear(SPATIAL_DIM,NUM_CLASSES)
        )
        self.domain=nn.Sequential(
            nn.Linear(SPATIAL_DIM,64),nn.ReLU(),nn.Dropout(0.20),
            nn.Linear(64,TOTAL_SUBJECTS)
        )
        self.proj=nn.Sequential(
            nn.Linear(SPATIAL_DIM,128),nn.ReLU(),nn.Linear(128,128)
        )

    def forward(self,x,lambda_grl=0.0):
        f=self.sinc(x)
        s=self.dgnn(f)
        t=self.temporal(s.mean(dim=1))
        z=self.se(t)
        cls=self.cls(z)
        dom=self.domain(grl(z,lambda_grl))
        proj=F.normalize(self.proj(z),p=2,dim=1)
        return cls,dom,proj

print("Model modules loaded.")

In [ ]:
# CELL 5 - LOSS + HELPERS
class SupConLoss(nn.Module):
    def __init__(self,tau=SUPCON_TEMP):
        super().__init__(); self.tau=tau
    def forward(self,features,labels):
        sim=(features@features.T)/self.tau
        labels=labels.view(-1,1)
        mask=(labels==labels.T).float()
        logits_mask=torch.ones_like(mask); logits_mask.fill_diagonal_(0)
        mask*=logits_mask
        logp=sim-torch.log((torch.exp(sim)*logits_mask).sum(1,keepdim=True)+1e-8)
        pos=mask.sum(1); valid=pos>0
        if not valid.any():
            return torch.zeros((),device=features.device,requires_grad=True)
        return -((mask*logp).sum(1)/(pos+1e-8))[valid].mean()

def get_test_subjects():
    if EVAL_MODE=="PROJECT":
        return [s for s in CURATED_TEST_POOL if s in AVAILABLE_SUBJECTS][:NUM_TEST_FOLDS]
    if EVAL_MODE=="RANDOM":
        rng=random.Random(SEED)
        return rng.sample(AVAILABLE_SUBJECTS,min(NUM_TEST_FOLDS,len(AVAILABLE_SUBJECTS)))
    if EVAL_MODE=="EXHAUSTIVE":
        return list(AVAILABLE_SUBJECTS)
    raise ValueError("Invalid EVAL_MODE")

def split_sources(src):
    src=list(src); rng=np.random.default_rng(SEED)
    rng.shuffle(src)
    n=max(1,int(len(src)*VAL_FRACTION))
    return src[n:],src[:n]

def aux_weights(epoch):
    if epoch<WARMUP_EPOCHS: return 0.0,0.0
    p=(epoch-WARMUP_EPOCHS)/max(1,EPOCHS-WARMUP_EPOCHS)
    ramp=2/(1+np.exp(-10*(p-0.5)))
    return DOMAIN_WEIGHT_MAX*ramp,SUPCON_WEIGHT_MAX*ramp

def evaluate_basic(model,loader):
    model.eval(); ytrue=[]; ypred=[]
    with torch.no_grad():
        for x,y,_ in loader:
            z,_,_=model(x.to(DEVICE),0.0)
            ytrue.extend(y.numpy()); ypred.extend(z.argmax(1).cpu().numpy())
    rep=classification_report(ytrue,ypred,labels=list(range(4)),
                              target_names=CLASS_NAMES,output_dict=True,zero_division=0)
    return accuracy_score(ytrue,ypred),rep["macro avg"]["f1-score"]

In [ ]:
# CELL 6 - SMOKE TESTS
m=S3MambaDA().to(DEVICE)
x=torch.randn(2,N_CHANNELS,N_SAMPLES,device=DEVICE)
with torch.no_grad():
    a,b,c=m(x,0.0)
print("Input:",tuple(x.shape))
print("Class:",tuple(a.shape))
print("Domain:",tuple(b.shape))
print("Projection:",tuple(c.shape))
assert tuple(a.shape)==(2,4)
assert tuple(b.shape)==(2,TOTAL_SUBJECTS)
assert tuple(c.shape)==(2,128)

with torch.no_grad():
    so=m.sinc(x)
print("Sinc:",tuple(so.shape))
assert tuple(so.shape)==(2,NUM_FILTERS,N_CHANNELS,N_SAMPLES)

print("✓ All smoke tests passed.")

In [ ]:
# CELL 7 - TRAIN ONE FOLD
def train_fold(model,train_loader,val_loader):
    ce=nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    de=nn.CrossEntropyLoss()
    sc=SupConLoss()
    opt=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS,eta_min=1e-5)

    best_f1=-1; best_epoch=1; best_state=None; hist=[]

    for epoch in range(EPOCHS):
        model.train()
        tw=cw=dw=sw=count=0
        dom_w,sc_w=aux_weights(epoch)
        if epoch<WARMUP_EPOCHS:
            grl_l=0.0
        else:
            p=(epoch-WARMUP_EPOCHS)/max(1,EPOCHS-WARMUP_EPOCHS)
            grl_l=2/(1+np.exp(-10*p))-1

        for x,y,s in train_loader:
            x,y,s=x.to(DEVICE),y.to(DEVICE),s.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            cls,dom,z=m(x,grl_l)
            lc=ce(cls,y); ld=de(dom,s); ls=sc(z,y)
            loss=lc+dom_w*ld+sc_w*ls
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
            opt.step()

            n=x.size(0); count+=n
            tw+=loss.detach().item()*n; cw+=lc.detach().item()*n
            dw+=ld.detach().item()*n; sw+=ls.detach().item()*n

        sch.step()
        va,vf=evaluate_basic(model,val_loader)
        rec={
            "epoch":epoch+1,"total_loss":tw/max(1,count),
            "classification_loss":cw/max(1,count),
            "domain_loss":dw/max(1,count),
            "supcon_loss":sw/max(1,count),
            "domain_weight":dom_w,"supcon_weight":sc_w,
            "lambda_grl":grl_l,"validation_accuracy":va,
            "validation_macro_f1":vf,
            "learning_rate":opt.param_groups[0]["lr"]
        }
        hist.append(rec)

        if vf>best_f1:
            best_f1=vf; best_epoch=epoch+1; best_state=copy.deepcopy(model.state_dict())

        if epoch==0 or (epoch+1)%10==0:
            print(
                f"Epoch {epoch+1:03d}/{EPOCHS} | "
                f"Loss={rec['total_loss']:.4f} | "
                f"Cls={rec['classification_loss']:.4f} | "
                f"Dom={rec['domain_loss']:.4f} | "
                f"SupCon={rec['supcon_loss']:.4f} | "
                f"ValF1={vf:.4f}"
            )

    model.load_state_dict(best_state)
    return model,pd.DataFrame(hist),best_epoch,best_f1

In [ ]:
# CELL 8 - AdaBN + FINAL TEST
def apply_adabn(model,loader,n_trials):
    model.eval()
    xs=[]; n=0
    for x,_,_ in loader:
        xs.append(x); n+=x.size(0)
        if n>=n_trials: break
    if not xs: return model
    target_x=torch.cat(xs,0)[:n_trials].to(DEVICE)
    bn=[m for m in model.modules() if isinstance(m,nn.modules.batchnorm._BatchNorm)]
    if not bn: return model
    for m in bn:
        m.reset_running_stats(); m.momentum=1.0; m.train()
    with torch.no_grad(): model(target_x,0.0)
    for m in bn:
        m.momentum=0.1; m.eval()
    model.eval(); return model

def final_test(model,loader):
    model.eval(); yt=[]; yp=[]; probs=[]; emb=[]
    with torch.no_grad():
        for x,y,_ in loader:
            cls,_,z=model(x.to(DEVICE),0.0)
            p=torch.softmax(cls,1); pred=p.argmax(1)
            yt.extend(y.numpy()); yp.extend(pred.cpu().numpy())
            probs.append(p.cpu().numpy()); emb.append(z.cpu().numpy())
    probs=np.concatenate(probs); emb=np.concatenate(emb)
    rep=classification_report(yt,yp,labels=list(range(4)),
                              target_names=CLASS_NAMES,output_dict=True,zero_division=0)
    metrics={
        "accuracy":accuracy_score(yt,yp),
        "kappa":cohen_kappa_score(yt,yp),
        "macro_precision":rep["macro avg"]["precision"],
        "macro_recall":rep["macro avg"]["recall"],
        "macro_f1":rep["macro avg"]["f1-score"],
    }
    return metrics,np.array(yt),np.array(yp),probs,emb

In [ ]:
# CELL 9 - MASTER EXPERIMENT
def run_experiment():
    start=time.time()
    targets=get_test_subjects()
    folds=[]; preds=[]; embeds=[]; histories=[]

    print("="*90)
    print("S3MambaDA V2")
    print("Mode:",EVAL_MODE)
    print("Targets:",targets)
    print("="*90)

    for fold,target in enumerate(targets,1):
        print(f"\nFOLD {fold}/{len(targets)} - TARGET S{target:03d}")

        candidates=[s for s in AVAILABLE_SUBJECTS if s!=target]
        rng=random.Random(SEED+target)
        sources=rng.sample(candidates,min(NUM_TRAIN_SUBJECTS,len(candidates)))
        train_subs,val_subs=split_sources(sources)

        train_ds=EEGMMIDB_Dataset(DATA_DIR,train_subs,training=True)
        val_ds=EEGMMIDB_Dataset(DATA_DIR,val_subs,training=False)
        test_ds=EEGMMIDB_Dataset(DATA_DIR,[target],training=False)

        dist=Counter(test_ds.labels)
        if set(dist)!={0,1,2,3}:
            raise RuntimeError(f"S{target:03d} missing class: {dist}")

        print("Trials:",len(train_ds),len(val_ds),len(test_ds))
        print("Target classes:",dist)

        train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=0)
        val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
        test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=0)

        model=S3MambaDA().to(DEVICE)
        model,hist,best_epoch,best_f1=train_fold(model,train_loader,val_loader)
        hist["fold"]=fold; hist["target_subject"]=target
        histories.extend(hist.to_dict("records"))

        model=apply_adabn(model,test_loader,len(test_ds))
        metrics,yt,yp,prob,emb=final_test(model,test_loader)

        folds.append({
            "fold":fold,"target_subject":target,
            "train_subjects":len(train_subs),
            "validation_subjects":len(val_subs),
            "train_trials":len(train_ds),
            "validation_trials":len(val_ds),
            "test_trials":len(test_ds),
            "best_epoch":best_epoch,
            "best_validation_f1":best_f1,
            **metrics
        })

        for i in range(len(yt)):
            row={"fold":fold,"target_subject":target,
                 "true_label":int(yt[i]),"pred_label":int(yp[i])}
            for c in range(4): row[f"prob_{c}"]=float(prob[i,c])
            preds.append(row)

            erow={"fold":fold,"target_subject":target,"true_label":int(yt[i])}
            for j in range(emb.shape[1]): erow[f"z_{j}"]=float(emb[i,j])
            embeds.append(erow)

        torch.save(
            {"model_state_dict":model.state_dict(),
             "target_subject":target,
             "best_epoch":best_epoch,
             "metrics":metrics},
            CHECKPOINT_DIR/f"fold_{fold:02d}_S{target:03d}.pth"
        )

        print(
            f"RESULT S{target:03d} | "
            f"Acc={metrics['accuracy']*100:.2f}% | "
            f"Kappa={metrics['kappa']:.4f} | "
            f"F1={metrics['macro_f1']:.4f}"
        )

    fold_df=pd.DataFrame(folds)
    predictions_df=pd.DataFrame(preds)
    embeddings_df=pd.DataFrame(embeds)
    history_df=pd.DataFrame(histories)

    fold_df.to_csv(RESULTS_DIR/"fold_metrics.csv",index=False)
    predictions_df.to_csv(RESULTS_DIR/"test_predictions.csv",index=False)
    embeddings_df.to_csv(RESULTS_DIR/"test_embeddings.csv",index=False)
    history_df.to_csv(RESULTS_DIR/"training_history.csv",index=False)

    summary={
        "evaluation_mode":EVAL_MODE,
        "target_subjects":targets,
        "folds":len(fold_df),
        "total_test_samples":len(predictions_df),
        "mean_accuracy":float(fold_df.accuracy.mean()),
        "std_accuracy":float(fold_df.accuracy.std(ddof=0)),
        "mean_kappa":float(fold_df.kappa.mean()),
        "std_kappa":float(fold_df.kappa.std(ddof=0)),
        "mean_macro_f1":float(fold_df.macro_f1.mean()),
        "std_macro_f1":float(fold_df.macro_f1.std(ddof=0)),
        "channels":N_CHANNELS,"samples":N_SAMPLES,
        "crop":[TMIN,TMAX],"device":str(DEVICE)
    }
    with open(RESULTS_DIR/"summary.json","w") as f: json.dump(summary,f,indent=4)

    print("\n"+"="*90)
    print(f"Accuracy: {summary['mean_accuracy']*100:.2f}% ± {summary['std_accuracy']*100:.2f}%")
    print(f"Kappa:    {summary['mean_kappa']:.4f} ± {summary['std_kappa']:.4f}")
    print(f"Macro-F1: {summary['mean_macro_f1']:.4f} ± {summary['std_macro_f1']:.4f}")
    print("Test samples:",summary["total_test_samples"])
    print(f"Runtime: {(time.time()-start)/60:.2f} min")
    print("="*90)

    return fold_df,predictions_df,embeddings_df,history_df,summary

print("Master experiment function ready.")

In [ ]:
# CELL 10 - RUN
fold_df,predictions_df,embeddings_df,history_df,summary=run_experiment()
display(fold_df)

In [ ]:
# CELL 11 - PAPER FIGURES: TRAINING CURVES
g=history_df.groupby("epoch")[[
    "total_loss","classification_loss","domain_loss","supcon_loss","validation_macro_f1"
]].mean()

fig,ax=plt.subplots(2,1,figsize=(10,10))
for col,label in [
    ("total_loss","Total"),
    ("classification_loss","Classification"),
    ("domain_loss","Domain"),
    ("supcon_loss","SupCon")
]:
    ax[0].plot(g.index,g[col],label=label)
ax[0].axvline(WARMUP_EPOCHS,ls="--",label="Warm-up end")
ax[0].set_title("Training Loss Components"); ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss")
ax[0].grid(alpha=.25); ax[0].legend()

ax[1].plot(g.index,g.validation_macro_f1,label="Validation Macro-F1",lw=2)
ax[1].axvline(WARMUP_EPOCHS,ls="--",label="Warm-up end")
ax[1].set_title("Validation Macro-F1"); ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Macro-F1")
ax[1].grid(alpha=.25); ax[1].legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR/"Fig01_Training_Validation.png",dpi=400,bbox_inches="tight")
plt.show()

In [ ]:
# CELL 12 - PAPER FIGURES: ACCURACY
a=fold_df.accuracy*100
fig,ax=plt.subplots(figsize=(11,6))
bars=ax.bar(fold_df.target_subject.astype(str),a)
ax.axhline(a.mean(),ls="--",label=f"Mean={a.mean():.2f}%")
ax.axhline(25,ls=":",label="Chance=25%")
ax.set_xlabel("Target Subject"); ax.set_ylabel("Accuracy (%)")
ax.set_title("Subject-Independent Accuracy"); ax.grid(axis="y",alpha=.25); ax.legend()
for b,v in zip(bars,a): ax.text(b.get_x()+b.get_width()/2,v+1,f"{v:.2f}%",ha="center")
plt.tight_layout(); plt.savefig(FIGURES_DIR/"Fig02_Fold_Accuracy.png",dpi=400,bbox_inches="tight"); plt.show()

In [ ]:
# CELL 13 - PAPER FIGURES: CONFUSION MATRIX
cm=confusion_matrix(predictions_df.true_label,predictions_df.pred_label,labels=[0,1,2,3])
cmn=cm.astype(float)/np.maximum(cm.sum(axis=1,keepdims=True),1)
fig,ax=plt.subplots(figsize=(8,7))
im=ax.imshow(cmn)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(CLASS_NAMES,rotation=20,ha="right"); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Normalized Four-Class Confusion Matrix")
for i in range(4):
    for j in range(4):
        ax.text(j,i,f"{cmn[i,j]*100:.1f}%",ha="center",va="center")
fig.colorbar(im,ax=ax)
plt.tight_layout(); plt.savefig(FIGURES_DIR/"Fig03_Confusion_Matrix.png",dpi=400,bbox_inches="tight"); plt.show()

In [ ]:
# CELL 14 - PAPER FIGURES: CLASS METRICS
rep=classification_report(predictions_df.true_label,predictions_df.pred_label,
                           labels=[0,1,2,3],target_names=CLASS_NAMES,
                           output_dict=True,zero_division=0)
p=[rep[c]["precision"] for c in CLASS_NAMES]
r=[rep[c]["recall"] for c in CLASS_NAMES]
f=[rep[c]["f1-score"] for c in CLASS_NAMES]
x=np.arange(4); w=.25
fig,ax=plt.subplots(figsize=(11,6))
ax.bar(x-w,p,w,label="Precision"); ax.bar(x,r,w,label="Recall"); ax.bar(x+w,f,w,label="F1")
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES,rotation=20,ha="right"); ax.set_ylim(0,1.05)
ax.set_ylabel("Score"); ax.set_title("Four-Class Precision, Recall and F1")
ax.grid(axis="y",alpha=.25); ax.legend()
plt.tight_layout(); plt.savefig(FIGURES_DIR/"Fig04_Class_Metrics.png",dpi=400,bbox_inches="tight"); plt.show()

In [ ]:
# CELL 15 - PAPER FIGURES: ROC/AUC
yt=predictions_df.true_label.to_numpy()
pr=predictions_df[[f"prob_{i}" for i in range(4)]].to_numpy()
fig,ax=plt.subplots(figsize=(9,7)); aucs=[]
for c in range(4):
    b=(yt==c).astype(int)
    if b.min()==b.max(): continue
    fpr,tpr,_=roc_curve(b,pr[:,c]); av=auc(fpr,tpr); aucs.append(av)
    ax.plot(fpr,tpr,lw=2,label=f"{CLASS_NAMES[c]} (AUC={av:.3f})")
ax.plot([0,1],[0,1],"--",lw=1,label="Chance")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("Four-Class One-vs-Rest ROC"); ax.grid(alpha=.25); ax.legend()
plt.tight_layout(); plt.savefig(FIGURES_DIR/"Fig05_ROC_AUC.png",dpi=400,bbox_inches="tight"); plt.show()
print("Mean AUC:",np.mean(aucs))

In [ ]:
# CELL 16 - PAPER FIGURES: t-SNE
zcols=[c for c in embeddings_df.columns if c.startswith("z_")]
X=embeddings_df[zcols].to_numpy(); y=embeddings_df.true_label.to_numpy()
perp=min(30,max(5,(len(X)-1)//3))
Z=TSNE(n_components=2,perplexity=perp,init="pca",learning_rate="auto",random_state=SEED).fit_transform(X)

fig,ax=plt.subplots(figsize=(10,8))
for c in range(4):
    m=y==c
    ax.scatter(Z[m,0],Z[m,1],s=15,alpha=.7,label=CLASS_NAMES[c])
ax.set_title("t-SNE of Learned Target-Domain Embeddings")
ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2"); ax.grid(alpha=.15); ax.legend()
plt.tight_layout(); plt.savefig(FIGURES_DIR/"Fig06_tSNE.png",dpi=400,bbox_inches="tight"); plt.show()

In [ ]:
# CELL 17 - PAPER FIGURES: LEARNED SINC BANDS
trained_model=fold_model
with torch.no_grad():
    low=SINC_MIN_FREQ+torch.sigmoid(trained_model.sinc.low_parameter/10.0)*(SINC_MAX_FREQ-SINC_MIN_FREQ-2.0)
    high=torch.minimum(
        low+1+F.softplus(trained_model.sinc.band_parameter),
        torch.tensor(SINC_MAX_FREQ,device=low.device,dtype=low.dtype)
    )
    high=torch.maximum(high,low+1)
low=low.cpu().numpy(); high=high.cpu().numpy(); order=np.argsort(low)

fig,ax=plt.subplots(figsize=(11,7))
for pos,idx in enumerate(order):
    ax.plot([low[idx],high[idx]],[pos,pos],lw=8)
    ax.text(high[idx]+.5,pos,f"{low[idx]:.2f}-{high[idx]:.2f} Hz",va="center")
ax.set_yticks(range(len(order))); ax.set_yticklabels([f"Filter {i+1}" for i in range(len(order))])
ax.set_xlabel("Frequency (Hz)"); ax.set_title("Learned Sinc Frequency Bands"); ax.grid(axis="x",alpha=.25)
plt.tight_layout(); plt.savefig(FIGURES_DIR/"Fig07_Learned_Sinc_Bands.png",dpi=400,bbox_inches="tight"); plt.show()

In [ ]:
# CELL 18 - FINAL PAPER TABLE + INTEGRITY CHECK
paper=fold_df.copy()
paper["Accuracy (%)"]=(paper.accuracy*100).round(2)
paper["Kappa"]=paper.kappa.round(4)
paper["Macro Precision"]=paper.macro_precision.round(4)
paper["Macro Recall"]=paper.macro_recall.round(4)
paper["Macro F1"]=paper.macro_f1.round(4)

paper=paper[[
    "fold","target_subject","Accuracy (%)","Kappa",
    "Macro Precision","Macro Recall","Macro F1","best_epoch"
]]
display(paper)
paper.to_csv(TABLES_DIR/"Table_Final_Results.csv",index=False)

print()
print("FINAL PAPER STATISTICS")
print(f"Accuracy : {summary['mean_accuracy']*100:.2f}% ± {summary['std_accuracy']*100:.2f}%")
print(f"Kappa    : {summary['mean_kappa']:.4f} ± {summary['std_kappa']:.4f}")
print(f"Macro-F1 : {summary['mean_macro_f1']:.4f} ± {summary['std_macro_f1']:.4f}")
print("Test samples:",summary["total_test_samples"])
print("Classes:",sorted(predictions_df.true_label.unique().tolist()))
assert sorted(predictions_df.true_label.unique().tolist())==[0,1,2,3]
print("✓ Four-class integrity check passed.")

In [ ]:
# CELL 19 - SAVE COMPLETE CONFIGURATION
config={
    "seed":SEED,
    "data_dir":DATA_DIR,
    "runs":RUNS,
    "channel_mode":CHANNEL_MODE,
    "channels":N_CHANNELS,
    "sampling_rate":FS,
    "crop":[TMIN,TMAX],
    "samples":N_SAMPLES,
    "classes":CLASS_NAMES,
    "filters":NUM_FILTERS,
    "sinc_kernel":SINC_KERNEL,
    "sinc_range":[SINC_MIN_FREQ,SINC_MAX_FREQ],
    "gru_hidden":GRU_HIDDEN,
    "gru_layers":GRU_LAYERS,
    "epochs":EPOCHS,
    "warmup_epochs":WARMUP_EPOCHS,
    "learning_rate":LEARNING_RATE,
    "weight_decay":WEIGHT_DECAY,
    "label_smoothing":LABEL_SMOOTHING,
    "domain_weight_max":DOMAIN_WEIGHT_MAX,
    "supcon_weight_max":SUPCON_WEIGHT_MAX,
    "supcon_temperature":SUPCON_TEMP,
    "augmentation":USE_AUGMENTATION,
    "evaluation_mode":EVAL_MODE,
    "target_subjects":get_test_subjects(),
    "device":str(DEVICE)
}
with open(RESULTS_DIR/"experiment_config.json","w") as f:
    json.dump(config,f,indent=4)
print("All results saved under:",RESULTS_DIR.resolve())